In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv

In [1]:
import asyncio
import sys
import dbzero as db0
from dotenv import load_dotenv
from statek.agents.agent import Agent
from statek.executors.job import Job, JobDef, JobStatus
from statek.pyenv import PyEnv
from statek.executors.utils import run_jobs_loop, unsuspend_jobs
from statek.future import FutureResult, temporal
from statek.exceptions import FutureError

In [2]:
# sample tools mocs

def add(a: int, b: int) -> int:
    """Adds two elements"""
    return a + b

def multiply(a: int, b: int) -> int:
    """multiply two elements"""
    return a * b

def exit(reason: str):
    print(reason)

In [3]:
# Counter to track check_condition calls

call_count = 0    
def check_condition(_):
    """Returns True after 2 calls, False before that."""
    global call_count
    call_count += 1
    sys.stderr.write(f"check_condition called: {call_count} time(s). Result {call_count > 2}\n"); sys.stderr.flush()
    
    return call_count > 2

def fetch_result(future_result):
    """Fetch the result value."""
    global call_count
    sys.stderr.write(f"Fetch_result called {call_count}\n"); sys.stderr.flush()
    if call_count > 2:
        sys.stderr.write("Returning result\n"); sys.stderr.flush()
        return 5
    else:
        sys.stderr.write("Throwing exception\n"); sys.stderr.flush()
        raise FutureError(future_result=future_result)


# Define temporal functions as globals
@temporal(complement=fetch_result, condition=check_condition)
def get_value():
    """Temporal function that returns a ready FutureResult."""
    return FutureResult(
        deps=None,
        state_num=0
    )

In [4]:
# Initialize dbzero
db0.init(".dbzero_data")
db0.open("test-unsuspend-jobs-loop")

In [5]:
max_jobs = 1

# Create agent and pyenv
agent = Agent(role="test", _system_prompt="""You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: \n{tools} \nYou only return one tool call at time e. g sum(2,5). Send only operation wrapped in print().
No comments just for eg: print(add(1,2)).""",_prompt_template="Solve this problem: {goal}", _tools=[add,multiply])
pyenv = PyEnv(local_state={
    "multiply":multiply,
    "add":add,
    "get_value":get_value
})
def start_jobs(capacity: int):
    global max_jobs
    if max_jobs <=0:
        return
    capacity = min(capacity, max_jobs)
    print(f"Creating jobs: {capacity}")
    for i in range(capacity):
        print(f"Creating {i} job")
        # Create job definition and job
        job_def = JobDef(
            agent=agent,
            job_params ={"goal":"3 * 5 + 4 * 2"},
            warmup_code=None
        )
        job = Job(
            job_def=job_def,
            model_family="test",
            model="openai/gpt-5",
            job_status=JobStatus.READY,
            py_env=pyenv
        )
        cls = job.__class__
        original = cls.set_status
        
        def wrapped(self, *args, **kwargs):
            sys.stderr.write(f"set_status was called {args} -- {kwargs} for {db0.uuid(self)}\n"); sys.stderr.flush()
            return original(self, *args, **kwargs)
        
        cls.set_status = wrapped
        max_jobs -= 1

In [6]:
from dotenv import load_dotenv
load_dotenv("./.env")


True

In [ ]:
result = await run_jobs_loop(25, "OPENROUTER", start_jobs)

set_status was called (<EnumValue JobStatus.STARTED>,) -- {} for JDOVTIZ57XIKTIEBUHFIABAN


Creating jobs: 1
Creating 0 job


Saving local context key: multiply with value : <function multiply at 0x7e9f311f2700>
Saving local context key: add with value : <function add at 0x7e9f33637240>
Saving local context key: get_value with value : <function get_value at 0x7e9f311f2b60>
Saving local context key: multiply with value : <function multiply at 0x7e9f311f2700>
Saving local context key: add with value : <function add at 0x7e9f33637240>
Saving local context key: get_value with value : <function get_value at 0x7e9f311f2b60>
Saving local context key: multiply with value : <function multiply at 0x7e9f311f2700>
Saving local context key: add with value : <function add at 0x7e9f33637240>
Saving local context key: get_value with value : <function get_value at 0x7e9f311f2b60>
Saving local context key: multiply with value : <function multiply at 0x7e9f311f2700>
Saving local context key: add with value : <function add at 0x7e9f33637240>
Saving local context key: get_value with value : <function get_value at 0x7e9f311f2b60>
